# Day 1 — Feel the tokens: cost, context and non-determinism
**Theory:** [day-01-what-is-an-agent.md](../theory/day-01-what-is-an-agent.md) · ⏱ ~45 min · Level: easy

**You will:**
1. Make your first Claude call and read `usage` (tokens = money)
2. Count tokens *before* sending a request
3. See how a big system prompt changes cost
4. Plot why agent cost grows **faster than linearly** with steps
5. See that the same prompt gives different answers (why agents need evals)

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("ANTHROPIC_API_KEY", UserSecretsClient().get_secret("ANTHROPIC_API_KEY"))
    except Exception as e:
        print("Add ANTHROPIC_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. Your first call: read the receipt
Every response carries `usage`: input tokens (what you sent) and output tokens (what the model wrote).
Output tokens cost ~5× more than input tokens.

In [ ]:
r = client.messages.create(
    model=CHEAP_MODEL, max_tokens=300,
    messages=[{"role": "user", "content": "In 2 sentences: what is an AI agent?"}],
)
tracker.add(CHEAP_MODEL, r.usage, label="first call")
print(text_of(r))
print("\nusage:", r.usage.input_tokens, "input tokens ·", r.usage.output_tokens, "output tokens")
print("stop_reason:", r.stop_reason)

## 2. Count tokens before you send
`count_tokens` is free and tells you exactly how big a request is. Useful for budgeting long prompts.

In [ ]:
short = [{"role": "user", "content": "What is an AI agent?"}]
policy_text = "\n".join(p.read_text() for p in sorted((REPO / "projects/shopease-support-agent/data/policies").glob("*.md")))
long = [{"role": "user", "content": policy_text * 5 + "\n\nWhat is the return window?"}]
for name, msgs in [("short", short), ("long", long)]:
    n = client.messages.count_tokens(model=MODEL, messages=msgs).input_tokens
    print(f"{name:>5}: {n:>6,} tokens  (~{len(msgs[0]['content'].split()):,} words)")

## 3. A big system prompt costs money on *every* call
Agents resend the system prompt on every loop step. Compare a tiny and a large one.

In [ ]:
big_system = "You are ShopEase support. Policies:\n" + policy_text * 8
for name, system in [("tiny system", "You are ShopEase support."), ("big system", big_system)]:
    r = client.messages.create(model=CHEAP_MODEL, max_tokens=100, system=system,
                               messages=[{"role": "user", "content": "Say hi in 5 words."}])
    cost = tracker.add(CHEAP_MODEL, r.usage, label=name)
    print(f"{name:<12} input={r.usage.input_tokens:>6,}  output={r.usage.output_tokens:>3}  ${cost:.5f}")

## 4. Why agent cost grows ~quadratically
At step *k* the prompt holds everything from steps 1..k-1. Total input = n·P₀ + a·n(n−1)/2.
(Prices: Opus 5.5 list prices per 1M tokens. Day 5 shows how caching flattens this curve.)

In [ ]:
import matplotlib.pyplot as plt

def agent_cost(n_steps, p0=3000, add_per_step=800, out_per_step=300, in_price=4.0, out_price=20.0):
    inp = n_steps * p0 + add_per_step * n_steps * (n_steps - 1) / 2
    return (inp * in_price + n_steps * out_per_step * out_price) / 1e6

steps = list(range(1, 41))
plt.figure(figsize=(6, 3.5))
plt.plot(steps, [agent_cost(n) for n in steps], label="actual (history grows)")
plt.plot(steps, [agent_cost(1) * n for n in steps], "--", label="if it were linear")
plt.xlabel("agent steps"); plt.ylabel("$ per run"); plt.legend(); plt.title("Agent cost vs steps"); plt.show()
for n in (5, 10, 30):
    print(f"{n:>2} steps: ${agent_cost(n):.3f}")

## 5. Same prompt, different answers
LLMs sample. That's why you can't test an agent with one try. You need evals (Day 11).

In [ ]:
answers = [ask("Give ONE creative name for an AI support agent for an online store. Only the name.") for _ in range(5)]
for a in answers:
    print("-", a.strip())
print("\nunique answers:", len(set(a.strip() for a in answers)), "/ 5")

## 6. Reliability math: why recovery matters
A chain of *n* steps that each succeed with probability *p* succeeds with *pⁿ*.

In [ ]:
for p in (0.95, 0.99):
    print(f"p={p}: " + "  ".join(f"{n} steps → {p ** n:.2f}" for n in (5, 10, 30)))

## 🆓 Free swap (optional)
Same question on a free provider. Needs `GEMINI_API_KEY` or `GROQ_API_KEY` (see SETUP.md).

In [ ]:
import os
from course_utils import free_chat
if os.getenv("GEMINI_API_KEY"):
    print(free_chat("gemini", "In 2 sentences: what is an AI agent?"))
elif os.getenv("GROQ_API_KEY"):
    print(free_chat("groq", "In 2 sentences: what is an AI agent?"))
else:
    print("No free-provider key set; skipping.")

## ✏️ Exercises
1. Change `add_per_step` in `agent_cost` to 3000 (big tool outputs). How much does a 30-step run cost now?
2. Compute: per-step accuracy 0.97, 20 steps, no recovery. What's the success rate?
3. Write one task where you'd use a **workflow** and one where you'd use an **agent**. Use the 4-point checklist from the theory note.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker`. Make a habit of checking cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None